# Ingest circuits.csv file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
   - Source File
   - Ingestion Timestamp
3. Write to bronze delta table   

In [0]:
%run "../00-common/01.environment-config"

In [0]:
%run "../00-common/02.bronze-helpers"

In [0]:
val source_file= landing_folder_path + "/circuits.csv"
val table_name= catalog_name + "." + bronze_schema + "." + "circuits"

In [0]:
import org.apache.spark.sql.types.{StructType,StructField,StringType,DoubleType}
val circuits_schema = StructType(Seq(
    StructField("circuitId",   StringType),
    StructField("url",         StringType),
    StructField("circuitName", StringType),
    StructField("lat",         DoubleType),
    StructField("long",        DoubleType),
    StructField("locality",    StringType),
    StructField("country",     StringType)
))


val circuits_df=spark.read.format("csv")
.option("header","true")
//.option("inferSchema","true")
.option("mode", "FAILFAST")
 .schema(circuits_schema)
.load(source_file)
//circuits_df.show(10,false)
display(circuits_df)
//circuits_df.printSchema()

In [0]:
val circuits_final_df=add_ingestion_metadata(circuits_df)
display(circuits_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
circuits_final_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
display(spark.table(table_name))
